# G3i on Colab: one block of the reversal gate on Gemma 3 12B

Runtime: **A100 40 GB** (Runtime > Change runtime type). Gemma 3 12B in bfloat16 is 24.4 GB of
weights; an L4 or T4 will not do. Set `BLOCK` below, run all cells, and when the last cell prints
`BLOCK_DONE` the record is in your Drive under `G3i/<block>/`. The calibration block runs first; the
test block only after the seed is drawn and committed, and the notebook refuses it otherwise.

In [ ]:
BLOCK = "calibration"   # or "test", only after the sealed config carries seeds.test
SEAL_COMMIT = "d0b73a43596694588459f5b6a6b41eda75815028"


In [ ]:
import subprocess, sys, json, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.56.1", "accelerate==1.14.0", "huggingface_hub", "safetensors"], check=True)
import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__, "bf16 ok:", torch.cuda.is_bf16_supported())

In [ ]:
# The repository at the sealed commit, so the harness is exactly the one the registration names.
if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", "-q", "https://github.com/ahb-sjsu/geometric-evaluation-theory", "/content/repo"], check=True)
subprocess.run(["git", "-C", "/content/repo", "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", "/content/repo", "checkout", "-q", "master"], check=True)   # the config may carry a later seed commit
subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
print(subprocess.run(["git", "-C", "/content/repo", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
cfg = json.load(open("/content/repo/experiments/G3i/flip_config_gemma12b.json"))
print("budgets", cfg["budgets"], "| seeds", cfg["seeds"])
if BLOCK == "test" and "test" not in cfg["seeds"]:
    raise SystemExit("REFUSED: no test seed in the sealed config; the test block runs only after it is drawn and committed")
if BLOCK == "calibration" and "test" in cfg["seeds"]:
    raise SystemExit("REFUSED: a test seed exists; the calibration block precedes it")

In [ ]:
# The judge's weights, from the ungated mirror at the registered revision, into a plain directory
# with the same STAGED.json manifest the NRP path writes, so the harness loads it by key.
from huggingface_hub import snapshot_download
m = cfg["model"]
dest = "/content/models/" + m["key"]
path = snapshot_download(m["model_id"], revision=m["revision"], local_dir=dest, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"])
json.dump({"model_id": m["model_id"], "revision": m["revision"]}, open(dest + "/STAGED.json", "w"))
print("staged", m["model_id"], m["revision"], "->", dest)

In [ ]:
# One block, G3d's harness unmodified. The output tree is the same as on the workstation and the cluster.
env = dict(os.environ, HF_HUB_OFFLINE="1", G3C_MODEL_ROOT="/content/models", PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
out = "/content/out/gemma12b"
r = subprocess.run([sys.executable, "flip.py", "run", "--config", "../G3i/flip_config_gemma12b.json", "--block", BLOCK, "--out", out],
                   cwd="/content/repo/experiments/G3d", env=env)
print("exit", r.returncode)
assert r.returncode == 0

In [ ]:
# The record to Drive: results.json, stimuli.json.gz, pairs.jsonl.gz.
from google.colab import drive
drive.mount("/content/drive")
import shutil
dst = "/content/drive/MyDrive/G3i/" + BLOCK
os.makedirs(dst, exist_ok=True)
for fn in os.listdir(f"{out}/{BLOCK}"):
    shutil.copy2(f"{out}/{BLOCK}/{fn}", dst)
print(os.listdir(dst))
print("BLOCK_DONE", BLOCK)